In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 08 · Audit, kill switches and anomaly detection

**Primer section:** §9 observability, audit, and governance · §4.5 bounding the envelope ·
threat model rows ASI08 (cascading failures), ASI10 (rogue agents).

The fifth idea in the primer: *observable by construction*. Every tool call leaves one structured
event that answers who asked, which agent acted, under whose authority, what exactly ran, who
approved it, and why policy allowed it. This notebook generates a few turns, queries the audit log
like an investigator, then exercises the governance loop's **revoke** step: a policy kill switch that
takes effect without a redeploy, revocation of a user's consent, and a small anomaly heuristic.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

from collections import Counter, defaultdict
from datetime import datetime

from agentsec.agents import REFUNDS, LocalStack, Step, reset_demo_state
from agentsec.audit import AuditLog
from agentsec.config import Settings
from agentsec.runtime import confirm, resume_after_auth, run_turn, seed_session

reset_demo_state()

USER = {"subject": "u-ana", "email": "ana@customer.example", "tenant": "acme"}
SCOPES = ["customers:read", "orders:read", "payments:refund", "email:send"]

audit = AuditLog()
stack = LocalStack.create(Settings(), audit=audit)
AGENT = stack.agent_id.spiffe_id

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

## 1. Generate some history

Two sessions: one ordinary support conversation with a read, a denied tool, an in-envelope refund and
an approved out-of-envelope refund; and one where a *different* user's session sees a denial.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="ana-1", user=USER, scopes=SCOPES)
stack.script(
    Step.call("lookup_customer", email="ana@customer.example"),
    Step.call("run_sql", query="select * from customers"),
    Step.call("issue_refund", order_id="O-5002", amount=35.0, currency="USD", reason="duplicate charge"),
    Step.call("issue_refund", order_id="O-5001", amount=120.0, currency="USD", reason="event cancelled"),
    Step.say("Refunded O-5002; O-5001 needs your approval."),
)
r = await run_turn(stack.runner, user_id="u-ana", session_id="ana-1", message="refund my orders please")
stack.script(Step.say("Refund on O-5001 issued."))
await confirm(stack.runner, user_id="u-ana", session_id="ana-1", pending=r.pending_confirmations[0], confirmed=True)

BEN = {"subject": "u-ben", "email": "ben@customer.example", "tenant": "acme"}
await seed_session(stack.runner, user_id="u-ben", session_id="ben-1", user=BEN, scopes=["customers:read"])  # no payments:refund
stack.script(Step.call("issue_refund", order_id="O-5003", amount=20.0, currency="USD", reason="x"), Step.say("I cannot refund that."))
await run_turn(stack.runner, user_id="u-ben", session_id="ben-1", message="refund O-5003")

print(len(audit.events()), "audit events;", len(REFUNDS), "refunds issued")
for row in audit.timeline():
    print(row)

## 2. Query the log like an investigator

`AuditLog` keeps an in-memory sink for notebooks (on GCP the same events land in Cloud Logging as
`jsonPayload` and can be routed to BigQuery). Every event carries the **dual identity**.

In [ ]:
print("events by this agent          :", len(audit.by_agent(AGENT)))
print("denials                       :", [(e.user, e.tool, e.reasons[0][:48]) for e in audit.denials()])
print("approvals (with approver)     :", [(e.tool, e.args_redacted.get('amount'), e.approver) for e in audit.events(lambda e: e.approver)])
print("destructive allows            :", [(e.user, e.tool, e.reasons) for e in audit.events(lambda e: e.event_type == 'tool.decision' and e.decision == 'allow' and e.tool == 'issue_refund')])

identities = Counter((short(e.agent), e.user, e.authority) for e in audit.events())
print("\ndual identity on every event  :")
for (agent, user, authority), n in identities.items():
    print(f"  agent={agent} user={user} authority={authority} × {n}")
assert all(e.agent == AGENT and e.authority == "delegated" for e in audit.events())

### What one event looks like on the wire

`to_dict()` redacts secrets and PII shapes before anything reaches a sink. The argument hash lets you
prove *which* call ran without storing the arguments at all.

In [ ]:
import json

approved = next(e for e in audit.events() if e.approver)
print(json.dumps(approved.to_dict(), indent=2, default=str)[:1200])

## 3. Kill switch #1: policy — effective immediately, no redeploy

Governance loop: register → permit → observe → **revoke** → evaluate. The engine reads the policy
object on every call, so removing the agent from a tool's allow list (or swapping in a stricter
`Policy`) takes effect on the very next tool call. On GCP the equivalent one-liners are an IAM deny
policy on the agent principal or a VPC-SC rule — nothing to redeploy either.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="ana-2", user=USER, scopes=SCOPES)

async def try_refund(session_id, amount=10.0):
    stack.script(Step.call("issue_refund", order_id="O-5002", amount=amount, currency="USD", reason="test"), Step.say("ok"))
    r = await run_turn(stack.runner, user_id="u-ana", session_id=session_id, message="refund")
    return next(t["response"] for t in r.tool_responses if t["name"] == "issue_refund")

before = await try_refund("ana-2")
print("before kill switch:", before.get("status") or before.get("error"))

saved_allow = list(stack.engine.policy.tools["issue_refund"].allow)
stack.engine.policy.tools["issue_refund"].allow = []          # <- the kill switch: nobody may refund
after = await try_refund("ana-2")
print("after kill switch :", after.get("error"), "|", after["reasons"][0])
assert before.get("status") == "issued" and after.get("error") == "policy_denied"

stack.engine.policy.tools["issue_refund"].allow = saved_allow  # restore
restored = await try_refund("ana-2")
print("after restore     :", restored.get("status"))
assert restored.get("status") == "issued"

A stricter alternative is to swap the whole policy object, e.g. a "read-only mode" policy that keeps
the read tools and drops everything else — same effect, versioned as a file.

In [ ]:
from agentsec.policy import Policy

read_only_policy = Policy.from_dict({
    "default": "deny",
    "principals": stack.policy.principals,
    "tools": {name: tp.model_dump(mode="json") for name, tp in stack.policy.tools.items() if tp.tier.value == "read"},
})
original_policy = stack.engine.policy
stack.engine.policy = read_only_policy
print("tools still allowed in read-only mode:", sorted(read_only_policy.tools))
denied = await try_refund("ana-2")
print("refund in read-only mode:", denied.get("error"), "|", denied["reasons"][0])
assert denied.get("error") == "policy_denied"
stack.engine.policy = original_policy

## 4. Kill switch #2: revoke a user's consent in the broker

Offboarding a user (or a user withdrawing consent) must cut the agent's delegated access to that
user's data immediately. `LocalAuthManager.revoke` drops the consent; the next `crm_lookup` pauses
for consent again instead of silently using a cached token.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="ana-3", user=USER, scopes=SCOPES)

stack.script(Step.call("crm_lookup", email="ana@customer.example"), Step.say("ok"))
r = await run_turn(stack.runner, user_id="u-ana", session_id="ana-3", message="check the CRM")
pending = r.pending_auth[0]
stack.auth_manager.finalize(auth_provider=stack.crm_provider, user_id="u-ana", consent_nonce=pending.consent_nonce)
stack.script(Step.say("ok"))
r2 = await resume_after_auth(stack.runner, user_id="u-ana", session_id="ana-3", pending=pending)
crm_content = next(t["response"] for t in r2.tool_responses if t["name"] == "crm_lookup")["content"]
print("with consent :", next(line for line in crm_content.splitlines() if line.startswith("CRM record")))

stack.auth_manager.revoke(auth_provider=stack.crm_provider, user_id="u-ana")   # <- the kill switch

stack.script(Step.call("crm_lookup", email="ana@customer.example"), Step.say("ok"))
r3 = await run_turn(stack.runner, user_id="u-ana", session_id="ana-3", message="check the CRM again")
print("after revoke : pending_auth =", len(r3.pending_auth), "| tool result:", next(t["response"] for t in r3.tool_responses if t["name"] == "crm_lookup"))
assert len(r3.pending_auth) == 1
print("broker log   :", [e.outcome for e in stack.auth_manager.access_log])

## 5. A small anomaly heuristic over the events

Agent Anomaly Detection on GCP does this at scale; the local version is a few lines: count destructive
*attempts* (allow, confirm or deny) per agent within any 60-second window and flag bursts. The budget in the policy
already caps destructive calls per invocation (2), so a hijacked model that loops sees denials — and
those denials are exactly the signal.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="burst", user=USER, scopes=SCOPES)
stack.script(*[Step.call("issue_refund", order_id="O-5002", amount=1.0, currency="USD", reason=f"loop {i}") for i in range(6)], Step.say("done"))
burst = await run_turn(stack.runner, user_id="u-ana", session_id="burst", message="refund")
outcomes = Counter((t["response"].get("status") or t["response"].get("error")) for t in burst.tool_responses)
print("burst turn outcomes:", dict(outcomes))

DESTRUCTIVE = {name for name, tp in stack.policy.tools.items() if tp.tier.value == "destructive"}

def destructive_bursts(log: AuditLog, *, window_s: int = 60, threshold: int = 5) -> list[tuple[str, str, int]]:
    """(agent, window start, count) for each agent with more than `threshold` destructive attempts in any `window_s` window."""
    by_agent: dict[str, list[datetime]] = defaultdict(list)
    for e in log.events(lambda e: e.event_type == "tool.decision" and e.tool in DESTRUCTIVE):
        by_agent[short(e.agent)].append(datetime.fromisoformat(e.ts))
    alerts = []
    for agent, times in by_agent.items():
        times.sort()
        for i, start in enumerate(times):
            n = sum(1 for t in times[i:] if (t - start).total_seconds() <= window_s)
            if n > threshold:
                alerts.append((agent, start.strftime("%H:%M:%S"), n))
                break
    return alerts

alerts = destructive_bursts(audit)
for agent, start, n in alerts:
    print(f"ALERT {agent}: {n} destructive attempts within 60 s of {start} (> 5)")
assert alerts, "expected the burst to trip the heuristic"

## On Google Cloud

* **Signals:** Cloud Audit Logs (both identities when delegated), Agent Observability traces, Agent
  Gateway telemetry, `agentsec-audit` structured logs → BigQuery sink (`infra/terraform/logging.tf`),
  Agent Anomaly Detection and Agent Threat Detection, the Agent Security Dashboard in SCC. Propagate
  `traceparent` through MCP `_meta` and A2A so one trace spans user → agent → tool.
* **Revoke:** an IAM deny policy on the agent principal or principalSet, a VPC-SC rule change, removing
  `roles/agentidentity.user` from an Auth Manager provider, or revoking a user's consent — none need a redeploy.
* **Evaluate:** run injection corpora and red-team suites against every release; treat instructions,
  tool lists and policies as versioned artifacts with review.

**In one sentence:** "I want every tool call to leave one event that answers who asked, which
agent acted, under whose authority, what exactly ran, who approved it, and why the policy allowed it —
then feed that into anomaly detection and keep a one-line kill switch: a deny policy on the agent's
principal, or pulling its allow-list entry, effective on the next call."